In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# Carregar as planilhas
df_base = pd.read_csv('Base.csv')
df_culturas_pragas = pd.read_csv('Culturas_e_pragas.csv')

# Visualizar as primeiras linhas das planilhas
print(df_base.head())
print(df_culturas_pragas.head())

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Modelo de embeddings com suporte a portugues e outros idiomas
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

def _first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

DIAG_BASE_COL = _first_existing_column(df_base, ["DIAGNÓSTICO", "DIAGNOSTICO", "Diagnostico", "Diagnóstico"])
DIAG_TRAT_COL = _first_existing_column(df_culturas_pragas, ["Categoria da Praga", "Diagnostico", "Diagnóstico"])
TRAT1_COL = _first_existing_column(df_culturas_pragas, ["Tratamento Nível 1 (Orgânico)", "Tratamento Nível 1"])
TRAT2_COL = _first_existing_column(df_culturas_pragas, ["Tratamento Nível 2 (Genérico)", "Tratamento Nível 2"])
TRAT3_COL = _first_existing_column(df_culturas_pragas, ["Tratamento Nível 3 (Agrotóxico Controlado)", "Tratamento Nível 3"])

if DIAG_BASE_COL is None or DIAG_TRAT_COL is None:
    raise ValueError("Colunas de diagnóstico não encontradas nas planilhas.")

diagnosticos_base = df_base[DIAG_BASE_COL].astype(str).tolist()
embeddings_diagnosticos = model.encode(
    diagnosticos_base,
    convert_to_tensor=True,
    normalize_embeddings=True
    )

def processar_respostas_com_embeddings(respostas):
    texto = " ".join([str(r).strip() for r in respostas if str(r).strip()])
    if not texto:
        return None
    return model.encode(texto, convert_to_tensor=True, normalize_embeddings=True)

def diagnostico_e_tratamento(respostas):
    embedding_respostas = processar_respostas_com_embeddings(respostas)
    if embedding_respostas is None:
        return {"error": "Respostas vazias ou inválidas"}

    similaridades = util.cos_sim(embedding_respostas, embeddings_diagnosticos)[0]
    indice = int(similaridades.argmax().item())
    diagnostico = diagnosticos_base[indice]

    tratamento_df = df_culturas_pragas[df_culturas_pragas[DIAG_TRAT_COL].astype(str) == diagnostico]
    if tratamento_df.empty:
        return {
            "diagnostico": diagnostico,
            "tratamento_nivel_1": "Tratamento não encontrado",
            "tratamento_nivel_2": "Tratamento não encontrado",
            "tratamento_nivel_3": "Tratamento não encontrado"
        }

    tratamento = tratamento_df.iloc[0]
    return {
        "diagnostico": diagnostico,
        "tratamento_nivel_1": str(tratamento[TRAT1_COL]) if TRAT1_COL else "Tratamento não encontrado",
        "tratamento_nivel_2": str(tratamento[TRAT2_COL]) if TRAT2_COL else "Tratamento não encontrado",
        "tratamento_nivel_3": str(tratamento[TRAT3_COL]) if TRAT3_COL else "Tratamento não encontrado"
    }

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok

# Inicializar o aplicativo Flask
app = Flask(__name__)

@app.route('/diagnostico', methods=['POST'])
def diagnostico():
    respostas = request.json.get('respostas')

    if not respostas:
        return jsonify({"error": "Respostas não fornecidas"}), 400

    # Chama a função para diagnosticar e retornar o tratamento
    resultado = diagnostico_e_tratamento(respostas)

    return jsonify(resultado)

# Endpoint básico para verificar se a API está ativa
@app.route('/')
def home():
    return jsonify({"message": "AgroScan API está ativa!"})

# Exposição opcional via ngrok no Colab
public_url = ngrok.connect(5000).public_url
print(f"URL pública: {public_url}")

# Iniciar a aplicação
app.run(port=5000)

In [ ]:
# Exemplo de chamada da API (executavel no Colab/Jupyter)

!curl -X POST "https://<ngrok-id>.ngrok.io/diagnostico" \
  -H "Content-Type: application/json" \
  -d '{"respostas": ["Milho", "Folhas amareladas", "Seca prolongada"]}'

In [ ]:
import gradio as gr

perguntas = [c for c in df_base.columns if c != DIAG_BASE_COL]

def interface_gradio(*respostas_usuario):
    respostas = list(respostas_usuario)
    resultado = diagnostico_e_tratamento(respostas)
    if "error" in resultado:
        return [resultado["error"], "-", "-", "-"]
    return [
        resultado.get("diagnostico", "-"),
        resultado.get("tratamento_nivel_1", "-"),
        resultado.get("tratamento_nivel_2", "-"),
        resultado.get("tratamento_nivel_3", "-")
    ]

inputs = [gr.Textbox(label=pergunta) for pergunta in perguntas]
outputs = [
    gr.Textbox(label="Diagnóstico"),
    gr.Textbox(label="Tratamento Nível 1 (Orgânico)"),
    gr.Textbox(label="Tratamento Nível 2 (Genérico)"),
    gr.Textbox(label="Tratamento Nível 3 (Agrotóxico Controlado)")
]

gr.Interface(
    fn=interface_gradio,
    inputs=inputs,
    outputs=outputs,
    title="AgroScan",
    description="Responda às perguntas para obter diagnóstico e recomendações de tratamento."
).launch()